# MinHash and LSH: Reuse Detection That Scales

**Computational Text Analysis Pack · Module 18 — Text Reuse and Intertextuality**

*Technique:* Signature sketching and locality-sensitive hashing; sub-quadratic search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/08-text-reuse-and-networks/18b_minhash_and_lsh.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

The previous notebook hit a wall: pairwise similarity over a real archive does
not finish. **MinHash** and **locality-sensitive hashing** are how the field
gets around it, and the trick is genuinely elegant.

MinHash replaces each document's shingle set with a short **signature** — a few
dozen numbers — with the property that the fraction of matching positions
estimates the Jaccard similarity. LSH then buckets signatures so that only
plausible pairs are ever compared.

Together they turn n² into something close to linear, which is the difference
between a method that works on a sample and one that works on an archive.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
def shingles(text, k=4):
    toks = tokenise(text)
    return {" ".join(toks[i:i + k]) for i in range(len(toks) - k + 1)}

def jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

# MinHash: hash every shingle with h different functions, keep the minimum each
# time. Two similar sets are likely to share their minimum, and how likely is
# exactly their Jaccard similarity.
MASKS = None

def build_hashes(n_hashes, seed=0):
    rng = np.random.default_rng(seed)
    return rng.integers(1, 2**61 - 1, size=(n_hashes, 2))

def signature(sh, hashes):
    if not sh:
        return np.full(len(hashes), np.iinfo(np.int64).max)
    base = np.array([hash(s) & 0xFFFFFFFF for s in sh], dtype=np.int64)
    a, b = hashes[:, 0][:, None], hashes[:, 1][:, None]
    return ((a * base[None, :] + b) % (2**31 - 1)).min(axis=1)

hashes = build_hashes(128, seed=3)
print("signature length:", len(hashes))

## Does the estimate actually work?

An approximation is only useful if you know its error. Compare estimated
similarity against true Jaccard on real pairs, at several signature lengths.

In [ ]:
docs = df.sample(n=180, random_state=12).reset_index(drop=True)
sets = [shingles(t) for t in docs["text"]]

test_pairs = [(i, j) for i in range(0, 60) for j in range(i + 1, 60)][:600]
true = np.array([jaccard(sets[i], sets[j]) for i, j in test_pairs])

print(f"{'signature length':>18}{'mean abs error':>18}{'corr with true':>18}")
print("-" * 54)
for n_h in [16, 32, 64, 128, 256]:
    hs = build_hashes(n_h, seed=3)
    sigs = [signature(s, hs) for s in sets[:60]]
    est = np.array([(sigs[i] == sigs[j]).mean() for i, j in test_pairs])
    print(f"{n_h:>18}{np.abs(est - true).mean():>18.4f}"
          f"{np.corrcoef(est, true)[0, 1]:>18.3f}")
print()
print("Error falls roughly as 1/sqrt(signature length). 128 hashes is 128")
print("numbers per document regardless of how long the document is — that is")
print("the compression that makes the archive tractable.")

In [ ]:
hs = build_hashes(128, seed=3)
sigs = [signature(s, hs) for s in sets[:60]]
est = np.array([(sigs[i] == sigs[j]).mean() for i, j in test_pairs])
plt.figure(figsize=(5.5, 5.5))
plt.scatter(true, est, s=10, alpha=.4)
plt.plot([0, max(true)], [0, max(true)], "--", color="crimson", lw=1)
plt.xlabel("true Jaccard")
plt.ylabel("MinHash estimate (128 hashes)")
plt.title("The sketch tracks the truth")
plt.tight_layout()

## LSH: never compare most pairs at all

Split each signature into **bands**. Two documents become candidates if they
match on *any* whole band. Similar documents match some band with high
probability; dissimilar ones almost never do — so the vast majority of pairs are
never compared.

The band count controls a threshold: the similarity above which pairs start
being caught. That is a parameter with a real meaning, not a tuning knob.

In [ ]:
def lsh_candidates(sigs, bands, rows):
    buckets = defaultdict(list)
    for idx, sig in enumerate(sigs):
        for b in range(bands):
            key = (b, tuple(sig[b * rows:(b + 1) * rows]))
            buckets[key].append(idx)
    cand = set()
    for members in buckets.values():
        if len(members) > 1:
            for i in range(len(members)):
                for j in range(i + 1, len(members)):
                    cand.add((min(members[i], members[j]),
                              max(members[i], members[j])))
    return cand

all_sigs = [signature(s, hs) for s in sets]
total_pairs = len(sets) * (len(sets) - 1) // 2
truth_pairs = {(i, j) for i in range(len(sets)) for j in range(i + 1, len(sets))
               if jaccard(sets[i], sets[j]) >= 0.3}

print(f"{'bands':>7}{'rows':>6}{'threshold':>12}{'candidates':>13}"
      f"{'% of all pairs':>16}{'recall':>9}")
print("-" * 63)
for bands, rows in [(8, 16), (16, 8), (32, 4), (64, 2)]:
    cand = lsh_candidates(all_sigs, bands, rows)
    thresh = (1 / bands) ** (1 / rows)
    recall = len(cand & truth_pairs) / len(truth_pairs) if truth_pairs else 1.0
    print(f"{bands:>7}{rows:>6}{thresh:>12.3f}{len(cand):>13,}"
          f"{len(cand)/total_pairs:>16.2%}{recall:>9.2f}")
print()
print(f"({len(truth_pairs)} genuinely similar pairs exist at Jaccard >= 0.3)")
print("More bands catches more, at the cost of examining more candidates. The")
print("threshold formula (1/b)^(1/r) tells you where the curve turns before you")
print("run anything.")

In [ ]:
# The payoff, stated as work avoided.
bands, rows = 32, 4
cand = lsh_candidates(all_sigs, bands, rows)
verified = [(i, j, jaccard(sets[i], sets[j])) for i, j in cand]
verified = sorted([v for v in verified if v[2] >= 0.3], key=lambda t: -t[2])
print(f"exhaustive comparisons avoided: {total_pairs - len(cand):,} "
      f"of {total_pairs:,} ({1 - len(cand)/total_pairs:.1%})")
print(f"reuse pairs confirmed         : {len(verified)}\n")
for i, j, s in verified[:5]:
    print(f"  {s:.3f}  [{docs.loc[i,'language']}] {docs.loc[i,'headline'][:60]}")
    print(f"         [{docs.loc[j,'language']}] {docs.loc[j,'headline'][:60]}")

### 🔧 Your turn 1

Set bands=64, rows=2 and look at the candidate count. At what point does LSH stop saving you anything?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

With very short bands almost everything collides, the candidate set approaches
the full pairwise set, and you have paid for hashing on top of the comparisons
you were trying to avoid. LSH is a *filter*, and a filter that passes everything
is overhead. The practical procedure is to fix the similarity threshold your
research question needs first, then solve (1/b)^(1/r) for a band configuration
near it — not to tune bands until the output looks nice.

</details>